# WasteWise — train a six-class detector in Colab

Upload **WasteWise_Lite.zip** (recommended) or the Complete ZIP. This notebook trains a separate YOLO detector on six broad classes. It does not fine-tune OWLv2 or teach exact peel names. In Colab select **Runtime → Change runtime type → GPU** before training. GPU availability is not guaranteed.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import zipfile, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.lower().endswith(".zip"))
with zipfile.ZipFile(archive) as z:
    for entry in z.infolist():
        path = PurePosixPath(entry.filename)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("Unsafe archive path")
    z.extractall("/content/wastewise_project")
project = Path("/content/wastewise_project/wastewise")
assert (project / "app.py").is_file(), "Upload the supplied project ZIP."
os.chdir(project)
print("Project:", project)


In [ ]:
%pip install -q -r requirements-training.txt
import torch
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())


## Prepare the actual dataset
The public archives contain 10,464 images with COCO boxes. Existing valid downloads are reused. Conversion creates a grouped split stratified by dominant class, records provenance, and skips invalid boxes. Review the audit and sample annotations before treating results as research evidence.


In [ ]:
import subprocess, sys
def run(*args):
    subprocess.run([sys.executable, *args], check=True)
run("scripts/download_dataset.py")
run("scripts/prepare_dataset.py")
run("scripts/audit_dataset.py")


## Inspect annotations
Do not assume every public annotation is correct. Repeat this cell with different images and classes.


In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display
import random
images = sorted(Path("data/garbage/yolo/train/images").glob("*.jpg"))
p = random.Random(234).choice(images)
im = Image.open(p).convert("RGB")
draw = ImageDraw.Draw(im)
names = ["biodegradable", "cardboard", "glass", "metal", "paper", "plastic"]
label = Path("data/garbage/yolo/train/labels") / (p.stem + ".txt")
for line in label.read_text().splitlines():
    cls, cx, cy, w, h = map(float, line.split())
    x1, y1 = (cx-w/2)*im.width, (cy-h/2)*im.height
    x2, y2 = (cx+w/2)*im.width, (cy+h/2)*im.height
    draw.rectangle((x1,y1,x2,y2), outline="red", width=2)
    draw.text((x1,y1), names[int(cls)], fill="red")
display(im)


## Train
Use validation data to choose settings. Keep the final test set separate. If CUDA is unavailable, request a GPU runtime; CPU training on the full dataset may take a long time.


In [ ]:
assert torch.cuda.is_available(), "Enable an available Colab GPU, or explicitly change this workflow to CPU."
run("scripts/train.py", "--epochs", "50", "--batch", "8", "--device", "0")


## Evaluate on the final test split
This generates actual metrics and plots. No accuracy is assumed in advance. Before claiming generalization, check capture-session overlap, near-duplicates and possible pretraining overlap.


In [ ]:
run("scripts/evaluate.py", "--weights", "models/custom/best.pt", "--device", "0")
print(Path("reports/test_metrics.json").read_text())


In [ ]:
import zipfile
result = Path("/content/WasteWise_trained_model.zip")
with zipfile.ZipFile(result, "w", zipfile.ZIP_DEFLATED) as z:
    for p in [Path("models/custom/best.pt"), Path("reports/test_metrics.json"), Path("data/garbage/yolo/split_manifest.json")]:
        z.write(p, str(p))
files.download(str(result))


## Use your trained model locally
Copy `models/custom/best.pt` into the application folder. Install `requirements-training.txt` using the local virtual environment, open WasteWise, choose **Custom YOLO**, and select that model path. It recognizes the six broad dataset labels. Specific organic object identities require an expanded, human-annotated training dataset.
